# CS406 ? Private segmentation training on Kaggle

Four-member course group. The remaining member identity is **2352xxxx (complete with verified details)**; the packaged `config/team.json` is the identity record.

Attach the **private** dataset produced by `scripts.prepare_kaggle`. This notebook verifies and restores its source/data archives, runs six fresh CompactUNet experiments, trains a fresh two-epoch pixel logistic baseline, then evaluates frozen selections together. It selects CUDA when PyTorch can use it, otherwise CPU. No credentials, Git clone, supplied pretrained weights, or Internet download are required.

The fixed protocol is 20 epochs, 128 ? 128 RGB inputs, 12 base channels, batch size 16, AdamW (learning rate 0.001, weight decay 0.0001). RGB preprocessing uses PIL bilinear resizing from native size to 256 ? 256, then 128 ? 128; binary masks use nearest-neighbor resizing. Seed 406 crosses BCE vs BCE + Dice with augmentation off/on; the full recipe repeats at seeds 407 and 408. Training uses Train, checkpoint/threshold selection uses Valid, and evaluation reads Test only after the relevant selections are frozen. The pixel baseline is trained after all six compact runs freeze; its existing trainer freezes its validation threshold before its own test evaluation.

Outputs below are initially empty. Only successful real execution creates logs, checkpoints, frozen selections and evaluation files. The original OSF DermaVQA (2024) release declares CC BY 4.0; mapping to the 2025 DermaVQA-DAS masks and this supplied local archive remains unconfirmed. Keep the dataset, notebook and outputs private.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import sys
import zipfile

INPUT_ROOT = Path(os.environ.get("CS406_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("CS406_WORK_ROOT", "/kaggle/working/cs406"))
manifest_matches = sorted(INPUT_ROOT.rglob("bundle-manifest.json"))
if len(manifest_matches) != 1:
    raise RuntimeError(f"Attach exactly one CS406 private bundle; found {len(manifest_matches)} manifests")
INPUT_DIR = manifest_matches[0].parent.resolve()
bundle_manifest = json.loads(manifest_matches[0].read_text(encoding="utf-8"))
if bundle_manifest.get("schema_version") != 1 or bundle_manifest.get("private") is not True:
    raise ValueError("Expected the supported private CS406 bundle")
source_meta = bundle_manifest["artifacts"]["source"]
bootstrap_records = [record for record in source_meta["files"] if record["path"] == "scripts/prepare_kaggle.py"]
if len(bootstrap_records) != 1:
    raise ValueError("Expected one recorded bootstrap source file")
bootstrap_record = bootstrap_records[0]
source_archive = INPUT_DIR / "cs406-source.zip"
if source_meta.get("file") != source_archive.name:
    raise ValueError("Unexpected source archive name")

def confined_bootstrap(path):
    relative = path.relative_to(INPUT_DIR)
    for index in range(1, len(relative.parts) + 1):
        component = INPUT_DIR.joinpath(*relative.parts[:index])
        if component.is_symlink() or getattr(component, "is_junction", lambda: False)():
            raise ValueError("Unsafe bootstrap input link")
    if not path.resolve().is_relative_to(INPUT_DIR):
        raise ValueError("Unsafe bootstrap input path")
    return path

confined_bootstrap(source_archive)
if source_archive.is_file():
    if source_archive.stat().st_size != source_meta["bytes"]:
        raise ValueError("Source archive byte-size mismatch")
    digest = hashlib.sha256()
    with source_archive.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != source_meta["sha256"]:
        raise ValueError("Source archive SHA-256 mismatch")
    # Read one exact bootstrap member; never extract unvalidated ZIP paths.
    with zipfile.ZipFile(source_archive) as archive:
        info = archive.getinfo("scripts/prepare_kaggle.py")
        if info.file_size > 256 * 1024:
            raise ValueError("Unexpected bootstrap size")
        bootstrap_code = archive.read(info)
else:
    # Kaggle ingestion may expand ZIPs at the dataset root or under the ZIP stem.
    for base in (INPUT_DIR / "cs406-source", INPUT_DIR):
        candidate = confined_bootstrap(base / "scripts/prepare_kaggle.py")
        if candidate.is_file():
            if candidate.stat().st_size > 256 * 1024:
                raise ValueError("Unexpected bootstrap size")
            bootstrap_code = candidate.read_bytes()
            break
    else:
        raise FileNotFoundError("Source ZIP and expanded bootstrap are both missing")
if len(bootstrap_code) != bootstrap_record["bytes"] or hashlib.sha256(bootstrap_code).hexdigest() != bootstrap_record["sha256"]:
    raise ValueError("Bootstrap source SHA-256 or byte-size mismatch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
bootstrap_path = WORK_ROOT / "prepare_kaggle_bootstrap.py"
bootstrap_path.write_bytes(bootstrap_code)
spec = importlib.util.spec_from_file_location("cs406_kaggle_bootstrap", bootstrap_path)
package = importlib.util.module_from_spec(spec)
spec.loader.exec_module(package)
restored = package.restore_bundle(INPUT_DIR, WORK_ROOT / "bundle")
SOURCE_DIR, DATA_DIR = restored["source_dir"], restored["data_dir"]
OUTPUT_DIR = WORK_ROOT / "outputs"
OUTPUT_DIR.mkdir()
CACHE_DIR = WORK_ROOT / "cache"
RUNS_DIR = OUTPUT_DIR / "runs"
os.chdir(SOURCE_DIR)
sys.path.insert(0, str(SOURCE_DIR))
print("Verified private bundle:", restored["manifest"]["dataset_ref"])
print("Input layouts:", restored["input_layouts"])
print("Code:", SOURCE_DIR, "Data:", DATA_DIR, "Outputs:", OUTPUT_DIR)


In [ ]:
import platform
import numpy as np
import PIL
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Optional supplied TransUNet reference, if separately attached as private input.
# It is never used to initialize the fresh CompactUNet experiments.
REFERENCE_CHECKPOINT = None
runtime = {
    "python": platform.python_version(), "torch": torch.__version__,
    "numpy": np.__version__, "pillow": PIL.__version__, "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "training_protocol": bundle_manifest["training_protocol"],
    "input_restoration": restored["input_layouts"],
}
(OUTPUT_DIR / "runtime.json").write_text(json.dumps(runtime, indent=2) + "\n", encoding="utf-8")
print(json.dumps(runtime, indent=2))


In [ ]:
import subprocess
import shutil

def run_logged(command, log_path):
    print("Running:", " ".join(map(str, command)), flush=True)
    try:
        with log_path.open("w", encoding="utf-8") as log:
            subprocess.run(list(map(str, command)), cwd=SOURCE_DIR, stdout=log,
                           stderr=subprocess.STDOUT, text=True, check=True)
    finally:
        if log_path.is_file():
            print(log_path.read_text(encoding="utf-8", errors="replace")[-12000:])

run_logged([
    sys.executable, "-m", "scripts.run_compact_experiments",
    "--data-dir", DATA_DIR, "--cache-dir", CACHE_DIR, "--output", RUNS_DIR,
    "--device", DEVICE, "--epochs", "20", "--cpu-threads", "4",
], OUTPUT_DIR / "training.log")
frozen_paths = sorted(RUNS_DIR.rglob("frozen-selection.json"))
if len(frozen_paths) != 6:
    raise RuntimeError(f"Expected six frozen experiment selections; found {len(frozen_paths)}")
for path in frozen_paths:
    selection = json.loads(path.read_text(encoding="utf-8"))
    history = json.loads((path.parent / "history.json").read_text(encoding="utf-8"))
    if selection.get("status") != "frozen" or len(history) != 20:
        raise RuntimeError(f"Incomplete experiment: {path.parent.name}")
cache_manifest_path = CACHE_DIR / "manifest.json"
if not cache_manifest_path.is_file():
    raise FileNotFoundError("Missing training cache provenance manifest")
shutil.copyfile(cache_manifest_path, OUTPUT_DIR / "training-cache-manifest.json")
print("All six 20-epoch selections are frozen; cache provenance is preserved.")


In [ ]:
# The compact training cell has already frozen every checkpoint and threshold.
if len(frozen_paths) != 6 or any(
    json.loads(path.read_text(encoding="utf-8")).get("status") != "frozen"
    for path in frozen_paths
):
    raise RuntimeError("Freeze all six compact experiments before training the pixel baseline")
PIXEL_DIR = OUTPUT_DIR / "pixel_classifier"
run_logged([
    sys.executable, "-m", "scripts.train_pixel_model",
    "--data-dir", DATA_DIR, "--output", PIXEL_DIR,
    "--epochs", "2", "--seed", "406",
], OUTPUT_DIR / "pixel-training.log")
pixel_manifest = json.loads((PIXEL_DIR / "manifest.json").read_text(encoding="utf-8"))
if pixel_manifest.get("status") != "complete" or not (PIXEL_DIR / "model.json").is_file():
    raise RuntimeError("Pixel baseline training did not complete")
print("Fresh two-epoch pixel baseline completed with validation-selected threshold.")


In [ ]:
evaluation_command = [
    sys.executable, "-m", "scripts.evaluate_completion",
    "--data-dir", DATA_DIR, "--runs-dir", RUNS_DIR,
    "--output", OUTPUT_DIR / "evaluation", "--device", DEVICE,
    "--pixel-model", PIXEL_DIR / "model.json",
]
if REFERENCE_CHECKPOINT is not None:
    reference_path = Path(REFERENCE_CHECKPOINT)
    if not reference_path.is_file():
        raise FileNotFoundError(reference_path)
    evaluation_command.extend(["--reference-checkpoint", reference_path])
run_logged(evaluation_command, OUTPUT_DIR / "evaluation.log")
evaluation_files = [p for p in (OUTPUT_DIR / "evaluation").rglob("*") if p.is_file()]
if not evaluation_files:
    raise RuntimeError("Evaluation exited without producing result files")
print("Evaluation files:", len(evaluation_files))


In [ ]:
import shutil

artifact_records = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        artifact_records.append({
            "path": path.relative_to(OUTPUT_DIR).as_posix(),
            "bytes": path.stat().st_size, "sha256": package.sha256_file(path),
        })
artifact_manifest = {
    "status": "completed", "device": DEVICE,
    "input_archives": {key: {field: entry[field] for field in ("file", "bytes", "sha256")}
                       for key, entry in bundle_manifest["artifacts"].items()},
    "files": artifact_records,
}
(OUTPUT_DIR / "artifact-manifest.json").write_text(json.dumps(artifact_manifest, indent=2) + "\n", encoding="utf-8")
archive_path = shutil.make_archive(str(WORK_ROOT.parent / "CS406_Kaggle_Artifacts"), "zip", OUTPUT_DIR)
print("Download from Kaggle Output:", archive_path)
print("Actual result files:", len(artifact_records))


Download `CS406_Kaggle_Artifacts.zip` and retain the Kaggle run URL/version. The archive contains the actual six training logs/checkpoints/selections and evaluation outputs, with a SHA-256 artifact manifest. Failed subprocesses stop execution; they do not produce a completed manifest. Verify that runtime.json records `cuda` for an actual GPU run. This source notebook contains no claimed metrics or simulated execution output.
